In [1]:
import getpass
import json
from typing import TypedDict, Literal
from langchain_core.messages import HumanMessage
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END

# Securely capture the API key as a string variable
anthropic_api_key = getpass.getpass("Enter your Anthropic API Key: ")

print("✅ Credentials locked in-memory. Ready to build graph architecture.")

✅ Credentials locked in-memory. Ready to build graph architecture.


In [5]:
# 1. DEFINE THE SHARED STATE GRAPH SCHEMA
class PlannerState(TypedDict):
    input_goal: str             # High-level goal
    current_plan: list[str]     # Step-by-step active checklist
    completed_tasks: list[str]  # Tasks successfully processed
    last_execution_error: str   # Error sent by executor layer
    final_output: str           # Summary report

# 2. INITIALIZE THE MODEL CORE USING ANTHROPIC
# Setting temperature=0 ensures precise, non-hallucinatory structural planning
llm_core = ChatAnthropic(
    model="claude-sonnet-4-6", 
    temperature=0,
    anthropic_api_key=anthropic_api_key
)

# 3. CONSTRUCT THE RUNTIME NODES
def planner_node(state: PlannerState) -> dict:
    print("\n📋 [NODE -> PLANNER]: Analyzing state matrix and organizing strategy roadmap...")
    
    # Initialization Path: Create initial plan
    if not state.get("current_plan"):
        prompt = f"""You are an elite QA Project Director. Break down the following high-level testing objective into a strict, sequential list of exactly 3 granular sub-tasks.
        Make sure one of the steps explicitly contains the word "payment" or "checkout" to test error scenarios.
        
        Objective: {state['input_goal']}
        
        Output your response ONLY as a valid, raw JSON array of strings containing exactly 3 items. Do not wrap it in markdown block tags or ticks.
        Example output format: ["Step 1 description", "Step 2 description", "Step 3 description"]"""
        
        response = llm_core.invoke([HumanMessage(content=prompt)])
        generated_plan = json.loads(response.content.strip())
        print(f"🔹 [PLANNER INITIALIZED PLAN]: {generated_plan}")
        return {"current_plan": generated_plan, "last_execution_error": ""}
    
    # Adaptive Re-Planning Path: Fix the crash
    else:
        print(f"⚠️ [PLANNER DETECTED FAULT]: Last worker error: '{state['last_execution_error']}'")
        print(f"🔄 [PLANNER RE-ROUTING]: Completed steps so far: {state['completed_tasks']}")
        
        prompt = f"""You are a QA Project Director tracking an automation run.
        Original Objective: {state['input_goal']}
        Completed Steps: {state['completed_tasks']}
        Current Remaining Plan: {state['current_plan']}
        Fatal Error Intercepted: {state['last_execution_error']}
        
        Your task is to rewrite the remaining plan checklist to bypass or resolve this error. Adjust descriptions to avoid the exact words "payment" or "checkout" so the simulated executor passes on retry.
        Output your updated remaining plan ONLY as a valid raw JSON array of strings. Do not use markdown tags."""
        
        response = llm_core.invoke([HumanMessage(content=prompt)])
        revised_plan = json.loads(response.content.strip())
        print(f"🔹 [PLANNER REVISED PLAN]: {revised_plan}")
        return {"current_plan": revised_plan, "last_execution_error": ""}

def executor_node(state: PlannerState) -> dict:
    next_task = state["current_plan"][0]
    print(f"\n🏃 [NODE -> EXECUTOR]: Pulling next task: '{next_task}'")
    
    # Simulated Failure Gate
    if "payment" in next_task.lower() or "checkout" in next_task.lower():
        print("💥 [ERROR -> EXECUTOR]: Element click validation timeout! Gateway failed to load (Stripe Token Error 403).")
        return {"last_execution_error": f"Task '{next_task}' crashed due to Stripe Token Error 403."}
    
    # Successful Run Execution Path
    print(f"✅ [NODE -> EXECUTOR]: Successfully executed and validated task: '{next_task}'")
    updated_completed = state.get("completed_tasks", []) + [next_task]
    remaining_plan = state["current_plan"][1:]
    
    return {"completed_tasks": updated_completed, "current_plan": remaining_plan}

def final_reporter_node(state: PlannerState) -> dict:
    print("\n📊 [NODE -> FINAL REPORTER]: Compiling deployment readiness summaries...")
    summary = f"All active objectives finalized. Total steps completed: {len(state['completed_tasks'])}. Adaptive system integration verified."
    return {"final_output": summary}

In [6]:
# 4. DEFINE THE CONTROL ROUTING LOGIC
def router_edge_logic(state: PlannerState) -> Literal["route_to_planner", "route_to_executor", "route_to_reporter"]:
    if state.get("last_execution_error"):
        print("🔀 [EDGE -> ROUTER]: Discrepancy caught! Rerouting to Planner for adaptive correction.")
        return "route_to_planner"
    
    if state.get("current_plan") and len(state["current_plan"]) > 0:
        print(f"🔀 [EDGE -> ROUTER]: Checklist contains {len(state['current_plan'])} pending items. Sending to Executor node.")
        return "route_to_executor"
    
    print("🔀 [EDGE -> ROUTER]: All tasks cleared. Routing to Final Report assembly.")
    return "route_to_reporter"

# 5. ASSEMBLE AND COMPILE THE GRAPH
workflow = StateGraph(PlannerState)
workflow.add_node("planner_brain", planner_node)
workflow.add_node("executor_worker", executor_node)
workflow.add_node("reporter_node", final_reporter_node)

workflow.add_edge(START, "planner_brain")
workflow.add_conditional_edges(
    "executor_worker",
    router_edge_logic,
    {
        "route_to_planner": "planner_brain",
        "route_to_executor": "executor_worker",
        "route_to_reporter": "reporter_node"
    }
)
workflow.add_edge("planner_brain", "executor_worker")
workflow.add_edge("reporter_node", END)

planner_agent_app = workflow.compile()
print("🎉 Stateful Planner-Executor Graph successfully compiled and locked.")

🎉 Stateful Planner-Executor Graph successfully compiled and locked.


In [7]:
macro_goal = "Perform an end-to-end regression test suite on our online storefront, focusing on user login and payment checkout validation workflows."

initial_input = {
    "input_goal": macro_goal,
    "current_plan": [],
    "completed_tasks": [],
    "last_execution_error": "",
    "final_output": ""
}

print("🚀 Launching Interactive Autonomous Planner-Executor Loop...")
print(f"Goal: {macro_goal}\n" + "-" * 70)

# Stream updates live as each individual node yields adjustments
events = planner_agent_app.stream(initial_input, stream_mode="updates")

for event in events:
    for node_name, state_update in event.items():
        print(f"\n🛑 [PAUSED] Node '{node_name}' finished executing.")
        print(f"Current State Update Delta: {json.dumps(state_update, indent=2)}")
        
        # Interactive Gate: Halts right here inside the notebook execution panel
        user_choice = input("\nPress Enter to ALLOW the graph to advance to the next step (or type 'quit' to halt): ")
        if user_choice.strip().lower() == "quit":
            print("❌ Execution manually terminated.")
            break

print("\n" + "=" * 70)
print("🏁 WORKFLOW ENDED. SYSTEM TRACE TERMINATED.")
print("=" * 70)

🚀 Launching Interactive Autonomous Planner-Executor Loop...
Goal: Perform an end-to-end regression test suite on our online storefront, focusing on user login and payment checkout validation workflows.
----------------------------------------------------------------------

📋 [NODE -> PLANNER]: Analyzing state matrix and organizing strategy roadmap...
🔹 [PLANNER INITIALIZED PLAN]: ['Verify user authentication workflows by testing valid login credentials, invalid password error handling, account lockout after failed attempts, and session token persistence across page navigation.', 'Execute end-to-end payment checkout validation by simulating successful transactions, declined card error scenarios, invalid CVV inputs, expired card edge cases, and confirming accurate order confirmation emails are triggered post-purchase.', 'Conduct post-checkout regression verification by validating order history updates in the user account dashboard, inventory decrement accuracy in the backend, and rollbac